# Fine-tune NuExtract-tiny-v1.5 with LoRA

Fine-tunes `numind/NuExtract-1.5-tiny` (Qwen2.5-0.5B) on the mealie-llm-server ingredient parsing dataset using LoRA.

**Runtime:** Use a GPU runtime (T4 is fine). Training takes ~2 minutes on T4.

**Output:** Q8_0 GGUF file (~507MB) for use with `MODEL_INGREDIENT_EXTRACTOR`.

In [ ]:
!pip install -q torch transformers peft trl datasets accelerate torchao>=0.16

In [ ]:
import json
import re
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "numind/NuExtract-1.5-tiny"
OUTPUT_DIR = Path("finetune-output")

# Hyperparameters
LORA_RANK = 16
LORA_ALPHA = 32
EPOCHS = 10
LR = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM = 2
MAX_SEQ_LENGTH = 512

In [ ]:
# Download JSONL from GitHub and strip ### Examples: section (v1 artifact not used by v1.5)
!wget -q -O ingredients_raw.jsonl "https://raw.githubusercontent.com/abyrne55/mealie-llm-server/main/tests/integration/ingredients.jsonl"

EXAMPLES_RE = re.compile(r"### Examples:\n.*?(?=### Text:)", re.DOTALL)
lines = open("ingredients_raw.jsonl").read().strip().splitlines()
with open("ingredients.jsonl", "w") as f:
    for line in lines:
        entry = json.loads(line)
        user_content = entry["messages"][0]["content"]
        user_content = EXAMPLES_RE.sub("", user_content)
        entry["messages"][0]["content"] = user_content
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Loaded {len(lines)} training examples")

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load and format dataset
JSONL_PATH = Path("ingredients.jsonl")
records = [json.loads(line) for line in JSONL_PATH.read_text().strip().splitlines()]
dataset = Dataset.from_list(records)
dataset = dataset.shuffle(seed=42)
dataset = dataset.map(
    lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)},
    remove_columns=dataset.column_names,
)
print(f"Dataset: {len(dataset)} examples")
print(f"Sample:\n{dataset[0]['text'][:300]}...")

In [ ]:
# Train
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    optim="adamw_torch",
    report_to="none",
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Merge LoRA and save
merged_model = model.merge_and_unload()
merged_path = OUTPUT_DIR / "merged"
merged_model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))
print(f"Merged model saved to {merged_path}")

In [ ]:
# Convert to GGUF Q8_0
!pip install -q gguf sentencepiece protobuf
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git 2>/dev/null || true

gguf_output = "nuextract-1.5-tiny-finetuned-q8_0.gguf"
!python llama.cpp/convert_hf_to_gguf.py {merged_path} --outfile {gguf_output} --outtype q8_0
print(f"\nGGUF saved to {gguf_output}")

In [ ]:
# Download the GGUF
from google.colab import files

files.download(gguf_output)
print("Copy to models/ and test with:")
print(f"MODEL_INGREDIENT_EXTRACTOR=models/{gguf_output} uv run pytest tests/integration/ -v")